[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science/blob/main/notebooks/Module_02_Data_Acquisition/M2_03_data_storage.ipynb)

# 💾 Module 02: Data Storage Patterns

**Purpose**: Learn best practices for storing, versioning, and documenting datasets  
**Module**: Module 02 - Data Acquisition  
**Author**: [Your Name]  
**Date**: 2026-01-15

---

## 📋 Overview

In this notebook, you will:
- [ ] Compare file formats (CSV, JSON, Parquet)
- [ ] Understand storage trade-offs (size, speed, compatibility)
- [ ] Implement data versioning strategies
- [ ] Set up proper `.gitignore` for data files
- [ ] Document data lineage and transformations
- [ ] Create efficient data storage pipelines

**Goal**: Establish professional data management practices!

**Estimated Time**: 60-90 minutes

---

## 📖 Part 1: Why Storage Matters

### The Data Storage Challenge

In data science projects, you'll work with:
- 📥 **Raw data** - Original, unmodified datasets
- 🔧 **Intermediate data** - Partially processed results
- 📊 **Final data** - Analysis-ready datasets
- 💾 **Model artifacts** - Trained models, predictions

**Poor storage practices lead to**:
- ❌ Lost data (no backups)
- ❌ Version confusion (which is latest?)
- ❌ Wasted space (inefficient formats)
- ❌ Slow processing (wrong format choice)
- ❌ Collaboration issues (inconsistent organization)

### Storage Principles

1. **🔒 Immutability** - Never modify raw data
2. **📝 Documentation** - Always document transformations
3. **🔢 Versioning** - Track changes over time
4. **⚡ Efficiency** - Choose appropriate formats
5. **🤝 Accessibility** - Make data discoverable

---

## 📊 File Format Comparison

| Format | Best For | Pros | Cons | Size | Speed |
|--------|----------|------|------|------|-------|
| **CSV** | Simple tabular data | Human-readable, universal | Large files, no types | ⭐⭐ | ⭐⭐ |
| **JSON** | Nested/hierarchical data | Flexible structure | Verbose, slow parsing | ⭐ | ⭐ |
| **Parquet** | Large datasets, analytics | Compressed, typed, fast | Binary (not readable) | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Feather** | Intermediate processing | Very fast I/O | Less compression | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **HDF5** | Multi-dimensional arrays | Efficient for NumPy | Complex, not portable | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |

**Rule of thumb**:
- 📁 Raw data → Keep original format (JSON, CSV)
- 🔧 Processed data → Parquet (best all-around)
- ⚡ Temporary data → Feather (fastest I/O)

---

## 🔧 Part 2: Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import os
import sys
import json
import time
from datetime import datetime
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
else:
    print("📍 Running locally")

# Add project root to path
project_root = os.path.abspath('../..' if 'notebooks' in os.getcwd() else '.')
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Part 3: Create Sample Dataset

Let's create a sample bike availability dataset to experiment with.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CREATE SAMPLE DATA
# ═══════════════════════════════════════════════════════════

# Create a realistic sample dataset
np.random.seed(42)

# Generate 30 days of hourly data (720 records)
hours = pd.date_range('2026-01-01', periods=720, freq='H')

# Simulate 10 bike stations
stations = [f'Station_{i:03d}' for i in range(1, 11)]

# Create data for each station
data_list = []
for station in stations:
    for timestamp in hours:
        # Add some realistic patterns
        hour = timestamp.hour
        day_of_week = timestamp.dayofweek
        
        # More bikes available during commute hours
        base_bikes = 10
        if hour in [8, 9, 17, 18]:
            base_bikes += np.random.randint(5, 15)
        
        # Fewer bikes on weekends
        if day_of_week >= 5:
            base_bikes = int(base_bikes * 0.7)
        
        data_list.append({
            'timestamp': timestamp,
            'station_id': station,
            'station_name': f'Amsterdam {station}',
            'bikes_available': base_bikes + np.random.randint(-5, 5),
            'docks_available': 20 - (base_bikes + np.random.randint(-5, 5)),
            'latitude': 52.37 + np.random.uniform(-0.05, 0.05),
            'longitude': 4.90 + np.random.uniform(-0.05, 0.05),
            'temperature_c': 15 + np.random.uniform(-10, 10),
            'precipitation_mm': max(0, np.random.exponential(0.5)),
            'wind_speed_kmh': max(0, np.random.normal(15, 5))
        })

df_sample = pd.DataFrame(data_list)

# Clean up negative values
df_sample['bikes_available'] = df_sample['bikes_available'].clip(lower=0)
df_sample['docks_available'] = df_sample['docks_available'].clip(lower=0)

print(f"✅ Created sample dataset with {len(df_sample):,} rows")
print(f"📊 Shape: {df_sample.shape}")
print(f"📅 Date range: {df_sample['timestamp'].min()} to {df_sample['timestamp'].max()}")
print(f"\n📋 First few rows:")
display(df_sample.head())

---

## 📁 Part 4: File Format Experiments

Let's compare different file formats by saving and loading the same data.

### Setup Test Directory

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. SETUP TEST DIRECTORY
# ═══════════════════════════════════════════════════════════

# Create temporary directory for experiments
TEST_DIR = Path('../../data/processed') if 'notebooks' in os.getcwd() else Path('data/processed')
TEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Test directory: {TEST_DIR}")
print(f"✅ Directory ready for format experiments")

### Experiment 1: Save in Different Formats

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. FORMAT COMPARISON - SAVE
# ═══════════════════════════════════════════════════════════

formats = {}

# 1. CSV Format
print("💾 Testing CSV format...")
csv_file = TEST_DIR / 'bike_data_test.csv'
start_time = time.time()
df_sample.to_csv(csv_file, index=False)
csv_time = time.time() - start_time
csv_size = csv_file.stat().st_size

formats['CSV'] = {
    'file': csv_file,
    'save_time': csv_time,
    'size_bytes': csv_size,
    'size_mb': csv_size / 1024 / 1024
}
print(f"  ✅ Saved in {csv_time:.3f}s, Size: {csv_size/1024:.1f} KB")

# 2. JSON Format
print("\n💾 Testing JSON format...")
json_file = TEST_DIR / 'bike_data_test.json'
start_time = time.time()
df_sample.to_json(json_file, orient='records', date_format='iso', indent=2)
json_time = time.time() - start_time
json_size = json_file.stat().st_size

formats['JSON'] = {
    'file': json_file,
    'save_time': json_time,
    'size_bytes': json_size,
    'size_mb': json_size / 1024 / 1024
}
print(f"  ✅ Saved in {json_time:.3f}s, Size: {json_size/1024:.1f} KB")

# 3. Parquet Format
print("\n💾 Testing Parquet format...")
parquet_file = TEST_DIR / 'bike_data_test.parquet'
start_time = time.time()
df_sample.to_parquet(parquet_file, index=False, compression='snappy')
parquet_time = time.time() - start_time
parquet_size = parquet_file.stat().st_size

formats['Parquet'] = {
    'file': parquet_file,
    'save_time': parquet_time,
    'size_bytes': parquet_size,
    'size_mb': parquet_size / 1024 / 1024
}
print(f"  ✅ Saved in {parquet_time:.3f}s, Size: {parquet_size/1024:.1f} KB")

# 4. Feather Format (if available)
try:
    print("\n💾 Testing Feather format...")
    feather_file = TEST_DIR / 'bike_data_test.feather'
    start_time = time.time()
    df_sample.to_feather(feather_file)
    feather_time = time.time() - start_time
    feather_size = feather_file.stat().st_size
    
    formats['Feather'] = {
        'file': feather_file,
        'save_time': feather_time,
        'size_bytes': feather_size,
        'size_mb': feather_size / 1024 / 1024
    }
    print(f"  ✅ Saved in {feather_time:.3f}s, Size: {feather_size/1024:.1f} KB")
except Exception as e:
    print(f"  ⚠️ Feather not available: {e}")

print("\n" + "=" * 60)
print("📊 Save Performance Summary:")
print("=" * 60)
for fmt, info in formats.items():
    print(f"{fmt:10s}: {info['save_time']:6.3f}s, {info['size_mb']:8.2f} MB")

### Experiment 2: Load from Different Formats

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. FORMAT COMPARISON - LOAD
# ═══════════════════════════════════════════════════════════

print("📂 Testing load performance...")
print("=" * 60)

# Test each format
for fmt, info in formats.items():
    file_path = info['file']
    
    start_time = time.time()
    
    if fmt == 'CSV':
        df_loaded = pd.read_csv(file_path, parse_dates=['timestamp'])
    elif fmt == 'JSON':
        df_loaded = pd.read_json(file_path)
    elif fmt == 'Parquet':
        df_loaded = pd.read_parquet(file_path)
    elif fmt == 'Feather':
        df_loaded = pd.read_feather(file_path)
    
    load_time = time.time() - start_time
    formats[fmt]['load_time'] = load_time
    
    print(f"{fmt:10s}: Loaded {len(df_loaded):,} rows in {load_time:.3f}s")

print("\n" + "=" * 60)
print("📊 Complete Performance Summary:")
print("=" * 60)
print(f"{'Format':<10} {'Save (s)':<10} {'Load (s)':<10} {'Size (MB)':<12} {'Compression':<12}")
print("-" * 60)

# Calculate compression ratios relative to CSV
csv_size = formats['CSV']['size_mb']
for fmt, info in formats.items():
    compression_ratio = (1 - info['size_mb'] / csv_size) * 100
    print(f"{fmt:<10} {info['save_time']:>8.3f}  {info['load_time']:>8.3f}  "
          f"{info['size_mb']:>10.2f}  {compression_ratio:>10.1f}%")

### Visualization: Format Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. VISUALIZE FORMAT COMPARISON
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('📊 File Format Comparison', fontsize=16, fontweight='bold')

format_names = list(formats.keys())
save_times = [formats[f]['save_time'] for f in format_names]
load_times = [formats[f]['load_time'] for f in format_names]
sizes_mb = [formats[f]['size_mb'] for f in format_names]

# 1. Save time comparison
axes[0].bar(format_names, save_times, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Save Performance\n(Lower is Better)')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(save_times):
    axes[0].text(i, v, f'{v:.3f}s', ha='center', va='bottom', fontweight='bold')

# 2. Load time comparison
axes[1].bar(format_names, load_times, color='darkorange', alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Load Performance\n(Lower is Better)')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(load_times):
    axes[1].text(i, v, f'{v:.3f}s', ha='center', va='bottom', fontweight='bold')

# 3. File size comparison
axes[2].bar(format_names, sizes_mb, color='green', alpha=0.7, edgecolor='black')
axes[2].set_ylabel('File Size (MB)')
axes[2].set_title('Storage Size\n(Lower is Better)')
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(sizes_mb):
    axes[2].text(i, v, f'{v:.2f}MB', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Recommendations
print("\n" + "=" * 60)
print("💡 Format Recommendations:")
print("=" * 60)
print("📁 Raw Data (preserve original):")
print("   → CSV/JSON - Keep in original format for traceability")
print("\n📊 Processed Data (analysis-ready):")
print("   → Parquet - Best balance of size, speed, and features")
print("\n⚡ Temporary/Intermediate Data:")
print("   → Feather - Fastest I/O for temporary workflows")
print("\n🤝 Sharing with Non-Technical Users:")
print("   → CSV - Universal compatibility, human-readable")

---

### 🧠 Task 4.1: Visualize Format Comparison (70% Scaffolding)

**Your Task**: Create visualizations comparing the file format performance.

**What you'll learn**: 
- How to interpret performance metrics
- Making data-driven format choices
- Creating comparison visualizations

**Requirements**:
1. Create a bar chart comparing file sizes across formats
2. Create a bar chart comparing save times across formats  
3. Add a recommendation based on your findings

**Hints**:
- Use the `formats` dictionary created above
- Try `plt.subplot(1, 2, 1)` for side-by-side plots
- Consider: Which format has the best size/speed trade-off?

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.1: FORMAT VISUALIZATION (70% SCAFFOLDING)
# ═══════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

# TODO: Create figure with 2 subplots side by side
# TODO: Plot 1 - Bar chart of file sizes
# TODO: Plot 2 - Bar chart of save times
# TODO: Add labels, titles, and grid
# TODO: Display the plots

# --- Your code here ---

# Expected output:
# - Two bar charts side by side
# - Left: File size comparison in MB
# - Right: Save time comparison in seconds
# - Clear labels and titles
# - A brief recommendation printed below

# Recommendation template:
# print("\n📊 Recommendation:")
# print(f"For this dataset size ({len(sample_data)} rows):")
# print("- Best for size: ???")
# print("- Best for speed: ???") 
# print("- Best balance: ???")

---

### 🧠 Task 4.2: Performance Testing with Larger Data (60% Scaffolding)

**Your Task**: Test read performance with a 10x larger dataset.

**What you'll learn**:
- How dataset size affects format choice
- Statistical performance testing
- When to use each format in production

**Requirements**:
1. Create a dataset with 7,200 rows (10x current size)
2. Save in all 4 formats
3. Test read performance 5 times each
4. Calculate mean and standard deviation
5. Generate performance report

**Provided structure** - Complete the TODOs:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.2: PERFORMANCE TESTING (60% SCAFFOLDING)
# ═══════════════════════════════════════════════════════════

import time
import numpy as np

# TODO: Create larger dataset (7200 rows = 10x current size)
# Hint: Use pd.concat([sample_data] * 10, ignore_index=True)

# TODO: Save in all 4 formats and measure time

# TODO: Test READ performance
# Run 5 trials for each format
# Calculate mean and std deviation

# --- Your code here ---

# Template for results:
results = {
    'csv': {'times': [], 'mean': 0, 'std': 0},
    'json': {'times': [], 'mean': 0, 'std': 0},
    'parquet': {'times': [], 'mean': 0, 'std': 0},
    'feather': {'times': [], 'mean': 0, 'std': 0}
}

# TODO: Run 5 read tests per format
# TODO: Calculate statistics
# TODO: Print formatted report

# Expected output:
# Performance Report (7200 rows)
# ========================================
# CSV:     0.XXX ± 0.XXX seconds
# JSON:    0.XXX ± 0.XXX seconds  
# Parquet: 0.XXX ± 0.XXX seconds
# Feather: 0.XXX ± 0.XXX seconds

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.1: FORMAT VISUALIZATION
# ═══════════════════════════════════════════════════════════

# TODO: Create side-by-side bar charts comparing formats

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: File Sizes
# Hint: Extract format names and sizes from the formats dict
format_names = list(formats.keys())
sizes_mb = [formats[fmt]['size_mb'] for fmt in format_names]

# Your code here: Create bar chart of sizes
# ax1.bar(...)
# ax1.set_xlabel(...)
# ax1.set_ylabel('Size (MB)')
# ax1.set_title(...)

# Chart 2: Save Times
save_times = [formats[fmt]['save_time'] for fmt in format_names]

# Your code here: Create bar chart of save times
# ax2.bar(...)
# ax2.set_xlabel(...)
# ax2.set_ylabel('Time (seconds)')
# ax2.set_title(...)

plt.tight_layout()
plt.show()

# Your analysis
print("\n" + "=" * 60)
print("📊 Format Recommendation:")
print("=" * 60)
# TODO: Based on your charts, which format would you recommend for:
# 1. Long-term storage?
# 2. Frequent read/write operations?
# 3. Sharing with non-technical users?
print("Your recommendations:")
print("  Long-term storage: ___________")
print("  Frequent I/O: ___________")
print("  Sharing: ___________")

---

### 🧠 Task 4.2: Deep Dive - Read Performance (60% Scaffolding)

**Your Task**: Test read performance with larger datasets to see format differences magnified.

**What you'll learn**:
- How dataset size affects format choice
- Practical performance testing
- When to use each format

**Requirements**:
1. Create a 10x larger dataset (7,200 rows)
2. Save in all formats
3. Test read performance 5 times and calculate average
4. Create a performance report

**Provided code structure** - Fill in the TODOs:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.2: PERFORMANCE TESTING
# ═══════════════════════════════════════════════════════════

# Step 1: Create larger dataset (10x)
# TODO: Multiply the hours to create 7,200 records
# Hint: Use pd.date_range with periods=7200

# Step 2: Save in all formats
large_formats = {}

for fmt_name in ['CSV', 'JSON', 'Parquet', 'Feather']:
    file_extension = fmt_name.lower()
    test_file = TEST_DIR / f'large_test.{file_extension}'
    
    # TODO: Add save logic for each format
    # if fmt_name == 'CSV':
    #     df_large.to_csv(test_file, index=False)
    # elif fmt_name == 'JSON':
    #     ...
    
    # TODO: Store file info
    # large_formats[fmt_name] = {'file': test_file}

# Step 3: Test read performance 5 times
NUM_TESTS = 5

for fmt_name, info in large_formats.items():
    read_times = []
    
    # TODO: Run 5 read tests and store times
    # for i in range(NUM_TESTS):
    #     start = time.time()
    #     if fmt_name == 'CSV':
    #         df = pd.read_csv(info['file'], parse_dates=['timestamp'])
    #     ...
    #     read_times.append(time.time() - start)
    
    # TODO: Calculate and store average
    # info['avg_read_time'] = np.mean(read_times)
    # info['std_read_time'] = np.std(read_times)
    pass

# Step 4: Create performance report
print("\n" + "=" * 70)
print("⚡ Performance Report - Large Dataset (7,200 rows)")
print("=" * 70)
# TODO: Print formatted results with average and standard deviation

---

## 🔢 Part 5: Data Versioning Strategies

### Versioning Approaches

1. **Timestamp-based**: `data_2026-01-15_143022.csv`
2. **Version numbers**: `data_v1.csv`, `data_v2.csv`
3. **Semantic versioning**: `data_v1.2.3.csv`
4. **Git LFS**: Version control for large files
5. **Data catalogs**: Metadata-driven (DVC, Pachyderm)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. VERSIONING IMPLEMENTATION
# ═══════════════════════════════════════════════════════════

class DataVersionManager:
    """
    Simple data versioning system with metadata tracking.
    """
    
    def __init__(self, base_dir):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.metadata_file = self.base_dir / 'versions.json'
        self.metadata = self._load_metadata()
    
    def _load_metadata(self):
        """Load version metadata from file."""
        if self.metadata_file.exists():
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return {'versions': []}
    
    def _save_metadata(self):
        """Save version metadata to file."""
        with open(self.metadata_file, 'w') as f:
            json.dump(self.metadata, f, indent=2)
    
    def save_version(self, df, name, description='', version=None):
        """
        Save a new version of the dataset.
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Data to save
        name : str
            Base name for the dataset
        description : str
            Description of changes in this version
        version : int, optional
            Specific version number (auto-increment if None)
        
        Returns:
        --------
        dict : Version metadata
        """
        # Determine version number
        if version is None:
            existing_versions = [v['version'] for v in self.metadata['versions'] 
                                if v['name'] == name]
            version = max(existing_versions, default=0) + 1
        
        # Create filename
        timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
        filename = f"{name}_v{version}_{timestamp}.parquet"
        filepath = self.base_dir / filename
        
        # Save data
        df.to_parquet(filepath, index=False)
        
        # Create version metadata
        version_info = {
            'name': name,
            'version': version,
            'filename': filename,
            'timestamp': timestamp,
            'datetime': datetime.now().isoformat(),
            'description': description,
            'rows': len(df),
            'columns': len(df.columns),
            'size_bytes': filepath.stat().st_size,
            'column_names': list(df.columns)
        }
        
        # Update metadata
        self.metadata['versions'].append(version_info)
        self._save_metadata()
        
        print(f"✅ Saved version {version} of '{name}'")
        print(f"   File: {filename}")
        print(f"   Rows: {len(df):,}, Size: {filepath.stat().st_size/1024:.1f} KB")
        
        return version_info
    
    def load_version(self, name, version=None):
        """
        Load a specific version of the dataset.
        
        Parameters:
        -----------
        name : str
            Dataset name
        version : int, optional
            Version number (latest if None)
        
        Returns:
        --------
        pandas.DataFrame : Loaded data
        """
        # Find matching versions
        matching = [v for v in self.metadata['versions'] if v['name'] == name]
        
        if not matching:
            raise ValueError(f"No versions found for dataset '{name}'")
        
        # Get specific or latest version
        if version is None:
            version_info = max(matching, key=lambda x: x['version'])
        else:
            version_info = next((v for v in matching if v['version'] == version), None)
            if version_info is None:
                raise ValueError(f"Version {version} not found for '{name}'")
        
        # Load data
        filepath = self.base_dir / version_info['filename']
        df = pd.read_parquet(filepath)
        
        print(f"✅ Loaded version {version_info['version']} of '{name}'")
        print(f"   Date: {version_info['datetime']}")
        print(f"   Description: {version_info['description']}")
        print(f"   Rows: {len(df):,}")
        
        return df
    
    def list_versions(self, name=None):
        """List all versions or versions for specific dataset."""
        if name:
            versions = [v for v in self.metadata['versions'] if v['name'] == name]
        else:
            versions = self.metadata['versions']
        
        if not versions:
            print(f"No versions found{' for ' + name if name else ''}")
            return
        
        print(f"\n📚 Available Versions{' for ' + name if name else ''}:")
        print("=" * 80)
        print(f"{'Name':<20} {'Ver':<5} {'Date':<20} {'Rows':<10} {'Description':<30}")
        print("-" * 80)
        
        for v in sorted(versions, key=lambda x: (x['name'], x['version'])):
            print(f"{v['name']:<20} {v['version']:<5} {v['datetime'][:19]:<20} "
                  f"{v['rows']:<10,} {v['description']:<30}")


# Example usage
print("🔧 Setting up version manager...")
vm = DataVersionManager(TEST_DIR)

# Save first version
print("\n💾 Saving version 1...")
vm.save_version(
    df_sample, 
    name='bike_availability',
    description='Initial dataset with 30 days of data'
)

# Make some changes and save version 2
df_modified = df_sample.copy()
df_modified['total_capacity'] = df_modified['bikes_available'] + df_modified['docks_available']

print("\n💾 Saving version 2...")
vm.save_version(
    df_modified,
    name='bike_availability',
    description='Added total_capacity column'
)

# List all versions
vm.list_versions()

### 🧠 Learner Task 1: Load and Compare Versions

**Your Task**: Load both versions and compare their differences.

Complete the code below:

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. LEARNER TASK 1: VERSION COMPARISON
# ═══════════════════════════════════════════════════════════

# TODO: Load version 1
df_v1 = vm.load_version('bike_availability', version=1)

# TODO: Load version 2 (latest)
df_v2 = vm.load_version('bike_availability')  # version=None gets latest

# TODO: Compare the two versions
print("\n" + "=" * 60)
print("🔍 Version Comparison:")
print("=" * 60)

# Hint: Compare shapes
print(f"Version 1 shape: {df_v1.shape}")
print(f"Version 2 shape: {df_v2.shape}")

# Hint: Compare columns
print(f"\nVersion 1 columns: {list(df_v1.columns)}")
print(f"Version 2 columns: {list(df_v2.columns)}")

# Hint: Find new columns in v2
new_cols = set(df_v2.columns) - set(df_v1.columns)
print(f"\nNew columns in v2: {new_cols}")

---

### 🧠 Task 4.1: Extend Version Manager (50% Scaffolding)

**Your Task**: Add a `delete_version()` method to the DataVersionManager class.

**What you'll learn**:
- Extending existing classes
- File system operations with pathlib
- Data lifecycle management

**Requirements**:
1. Implement method that deletes a specific version
2. Remove file from disk
3. Update metadata
4. Add error handling
5. Test by deleting version 1

**Method signature provided** - Implement the logic:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.1: EXTEND VERSION MANAGER (50% SCAFFOLDING)
# ═══════════════════════════════════════════════════════════

# TODO: Add this method to the DataVersionManager class above
# Then copy the entire class here and test it

class DataVersionManager:
    """Extended version with delete capability"""
    
    # ... (copy __init__, save_version, get_latest methods from above)
    
    def delete_version(self, name: str, version: int):
        """
        Delete a specific version of a dataset.
        
        TODO: Implement this method
        Requirements:
        1. Check if dataset exists in metadata
        2. Check if version exists
        3. Delete file from disk using Path.unlink()
        4. Remove from metadata dictionary
        5. Save updated metadata
        6. Print confirmation message
        
        Hint: Use try/except for error handling
        """
        # --- Your code here ---
        pass

# TODO: Test the delete functionality
# 1. Create new manager instance
# 2. Save a few versions
# 3. Delete version 1
# 4. Verify it's gone from both disk and metadata

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 5.1: EXTEND VERSION MANAGER
# ═══════════════════════════════════════════════════════════

def delete_version(self, name, version):
    """
    Delete a specific version of a dataset.
    
    Parameters:
    -----------
    name : str
        Dataset name
    version : int
        Version number to delete
    """
    # TODO: Find the version in metadata
    # Hint: Use list comprehension to find matching version
    # version_info = next((v for v in self.metadata['versions'] 
    #                     if v['name'] == name and v['version'] == version), None)
    
    # TODO: Check if version exists
    # if version_info is None:
    #     raise ValueError(f"Version {version} of '{name}' not found")
    
    # TODO: Delete the file
    # filepath = self.base_dir / version_info['filename']
    # if filepath.exists():
    #     filepath.unlink()  # Delete file
    
    # TODO: Remove from metadata
    # self.metadata['versions'] = [v for v in self.metadata['versions']
    #                              if not (v['name'] == name and v['version'] == version)]
    # self._save_metadata()
    
    # TODO: Print confirmation
    # print(f"✅ Deleted version {version} of '{name}'")
    # print(f"   File: {version_info['filename']}")
    
    pass

# Add method to the class
DataVersionManager.delete_version = delete_version

# Test your implementation
print("🧪 Testing delete_version...")
print("\nBefore deletion:")
vm.list_versions('bike_availability')

# TODO: Delete version 1
# vm.delete_version('bike_availability', 1)

print("\nAfter deletion:")
# vm.list_versions('bike_availability')

---

## 📝 Part 6: Data Documentation

### Documentation Best Practices

Always document:
1. **Source** - Where did the data come from?
2. **Date** - When was it collected?
3. **Transformations** - What changes were made?
4. **Schema** - What do the columns mean?
5. **Quality** - Are there known issues?

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. DATA DOCUMENTATION
# ═══════════════════════════════════════════════════════════

def create_data_documentation(df, dataset_name, source, description):
    """
    Create comprehensive data documentation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Dataset to document
    dataset_name : str
        Name of the dataset
    source : str
        Data source
    description : str
        Dataset description
    
    Returns:
    --------
    dict : Documentation metadata
    """
    doc = {
        'dataset_name': dataset_name,
        'description': description,
        'source': source,
        'created_date': datetime.now().isoformat(),
        'shape': {
            'rows': len(df),
            'columns': len(df.columns)
        },
        'columns': {},
        'data_types': {},
        'missing_values': {},
        'date_range': {},
        'file_info': {}
    }
    
    # Document each column
    for col in df.columns:
        doc['columns'][col] = {
            'dtype': str(df[col].dtype),
            'non_null_count': int(df[col].notna().sum()),
            'null_count': int(df[col].isna().sum()),
            'null_percentage': float(df[col].isna().sum() / len(df) * 100)
        }
        
        # Add statistics for numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            doc['columns'][col].update({
                'min': float(df[col].min()),
                'max': float(df[col].max()),
                'mean': float(df[col].mean()),
                'median': float(df[col].median()),
                'std': float(df[col].std())
            })
        
        # Add unique values for categorical columns
        if df[col].dtype == 'object' or df[col].nunique() < 50:
            doc['columns'][col]['unique_values'] = int(df[col].nunique())
            doc['columns'][col]['sample_values'] = df[col].dropna().unique()[:5].tolist()
    
    # Document date range if timestamp column exists
    if 'timestamp' in df.columns:
        doc['date_range'] = {
            'start': df['timestamp'].min().isoformat(),
            'end': df['timestamp'].max().isoformat(),
            'duration_days': (df['timestamp'].max() - df['timestamp'].min()).days
        }
    
    return doc


# Create documentation for our sample data
print("📝 Creating data documentation...")
documentation = create_data_documentation(
    df_sample,
    dataset_name='amsterdam_bike_availability',
    source='Simulated data for demonstration',
    description='Bike availability data with weather information for Amsterdam stations'
)

# Save documentation
doc_file = TEST_DIR / 'bike_availability_documentation.json'
with open(doc_file, 'w') as f:
    json.dump(documentation, f, indent=2, default=str)

print(f"✅ Documentation saved: {doc_file}")

# Display summary
print("\n" + "=" * 60)
print("📊 Dataset Documentation Summary:")
print("=" * 60)
print(f"Dataset: {documentation['dataset_name']}")
print(f"Description: {documentation['description']}")
print(f"Rows: {documentation['shape']['rows']:,}")
print(f"Columns: {documentation['shape']['columns']}")
print(f"\nDate Range: {documentation['date_range']['start'][:10]} to {documentation['date_range']['end'][:10]}")
print(f"Duration: {documentation['date_range']['duration_days']} days")

print("\n📋 Column Summary:")
for col, info in list(documentation['columns'].items())[:5]:  # Show first 5 columns
    print(f"\n  {col}:")
    print(f"    Type: {info['dtype']}")
    print(f"    Non-null: {info['non_null_count']:,}")
    if 'mean' in info:
        print(f"    Mean: {info['mean']:.2f}")

---

### 🧠 Task 6.1: Create Custom Documentation (40% Scaffolding)

**Your Task**: Document your latest dataset version with README and metadata.

**What you'll learn**:
- Professional data documentation standards
- Schema documentation best practices
- Making data discoverable

**Requirements**:
1. Load latest version from version manager
2. Create detailed README.md file
3. Generate metadata JSON with statistics
4. Document all transformations applied

**Template provided** - Customize for your data:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 6.1: CUSTOM DOCUMENTATION (40% SCAFFOLDING)
# ═══════════════════════════════════════════════════════════

# TODO: Load your latest dataset version

# TODO: Create README.md
readme_template = """
# Dataset: [Your Dataset Name]

## Overview
[Brief description of what this data contains]

## Schema
| Column | Type | Description | Example |
|--------|------|-------------|---------|
| station_id | int | Unique identifier | 123 |
| ... | ... | ... | ... |

## Statistics
- Total Records: ???
- Date Range: ??? to ???
- Missing Values: ???

## Transformations Applied
1. [List any changes made to the data]
2. ...

## Usage
```python
import pandas as pd
df = pd.read_parquet('bike_data_v3.parquet')
```

## Contact
Generated on: {date}
"""

# TODO: Write README to file

# TODO: Create metadata.json with actual statistics
metadata = {
    'dataset_name': 'bike_availability',
    'version': 3,
    'created_date': 'YYYY-MM-DD',
    'row_count': 0,  # TODO: Get actual count
    'columns': [],   # TODO: List column names
    'missing_values': {},  # TODO: Calculate missing counts
    'data_types': {}  # TODO: Get dtypes
}

# TODO: Write metadata to JSON file

# TODO: Print confirmation
print("✅ Documentation created:")
print("  - README.md")
print("  - metadata.json")

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 6.1: CUSTOM DOCUMENTATION
# ═══════════════════════════════════════════════════════════

# Step 1: Load latest version
# TODO: Load the latest version from your version manager
# df_latest = vm.load_version('bike_availability')

# Step 2: Create README content
readme_template = """# Amsterdam Bike Availability Dataset

## Overview
TODO: Describe what this dataset contains

## Source
- **Original data**: TODO
- **Collection period**: TODO
- **Update frequency**: TODO

## Schema

| Column | Type | Description |
|--------|------|-------------|
| timestamp | datetime | TODO |
| station_id | string | TODO |
| bikes_available | int | TODO |
| docks_available | int | TODO |
| total_capacity | int | TODO (if applicable) |

## Data Quality

- **Missing values**: TODO: Describe any missing data
- **Outliers**: TODO: Note any unusual values
- **Validation**: TODO: What checks were performed?

## Transformations Applied

1. TODO: List transformations
2. TODO: ...

## Usage Example

```python
import pandas as pd

# Load data
df = pd.read_parquet('bike_availability_v2_latest.parquet')

# Basic analysis
print(df.describe())
```

## Contact
**Maintainer**: Your Name  
**Last Updated**: TODO
"""

# TODO: Save README
# with open(TEST_DIR / 'README.md', 'w') as f:
#     f.write(readme_template)

# Step 3: Create metadata JSON
# TODO: Use the create_data_documentation function above
# metadata = create_data_documentation(
#     df_latest,
#     dataset_name='amsterdam_bike_availability',
#     source='Your source here',
#     description='Your description here'
# )

# TODO: Save metadata
# with open(TEST_DIR / 'metadata.json', 'w') as f:
#     json.dump(metadata, f, indent=2)

print("✅ Documentation created!")
print(f"   README.md: {TEST_DIR / 'README.md'}")
print(f"   metadata.json: {TEST_DIR / 'metadata.json'}")

---

## 🔒 Part 7: .gitignore Best Practices

### What NOT to Commit to Git

- ❌ Large data files (> 100MB)
- ❌ Raw data downloads
- ❌ API keys and credentials
- ❌ Personal/sensitive information
- ❌ Temporary/cache files

### What TO Commit

- ✅ Code and notebooks
- ✅ Small sample datasets (< 10MB)
- ✅ Data acquisition scripts
- ✅ Documentation and metadata
- ✅ README files explaining data sources

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. GITIGNORE RECOMMENDATIONS
# ═══════════════════════════════════════════════════════════

gitignore_content = """# Data Science Project .gitignore

# Large Data Files
data/raw/*.csv
data/raw/*.json
data/raw/*.parquet
data/processed/*.csv
data/processed/*.parquet

# Keep sample data and documentation
!data/raw/sample_*.csv
!data/raw/README.md
!data/processed/README.md

# API Keys and Credentials
.env
.env.local
*.key
credentials.json
secrets.json

# Jupyter Notebook Checkpoints
.ipynb_checkpoints/
*/.ipynb_checkpoints/*

# Python Cache
__pycache__/
*.py[cod]
*$py.class
*.so

# Virtual Environments
venv/
env/
ENV/
.venv

# IDE Settings
.vscode/
.idea/
*.swp
*.swo
*~

# OS Files
.DS_Store
Thumbs.db
desktop.ini

# Model Artifacts (large files)
models/*.h5
models/*.pkl
models/*.joblib

# Temporary Files
*.tmp
*.temp
temp_*

# Large Results
results/*.png
results/*.jpg
!results/summary_*.png  # Keep summary images
"""

print("📝 Recommended .gitignore content:")
print("=" * 60)
print(gitignore_content)
print("=" * 60)

# Check if .gitignore exists in project root
gitignore_path = Path(project_root) / '.gitignore'
if gitignore_path.exists():
    print(f"\n✅ .gitignore file exists at: {gitignore_path}")
    print("   Review and update it with the recommendations above")
else:
    print(f"\n⚠️ No .gitignore file found at: {gitignore_path}")
    print("   Consider creating one with the recommendations above")

---

## 📊 Part 8: Data Catalog

Create a data catalog to track all datasets in your project.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. DATA CATALOG
# ═══════════════════════════════════════════════════════════

class DataCatalog:
    """
    Manage a catalog of all datasets in the project.
    """
    
    def __init__(self, catalog_file):
        self.catalog_file = Path(catalog_file)
        self.catalog = self._load_catalog()
    
    def _load_catalog(self):
        """Load catalog from file."""
        if self.catalog_file.exists():
            with open(self.catalog_file, 'r') as f:
                return json.load(f)
        return {'datasets': []}
    
    def _save_catalog(self):
        """Save catalog to file."""
        with open(self.catalog_file, 'w') as f:
            json.dump(self.catalog, f, indent=2)
    
    def register_dataset(self, name, path, description, source, 
                        tags=None, metadata=None):
        """Register a new dataset in the catalog."""
        
        path = Path(path)
        
        dataset_info = {
            'name': name,
            'path': str(path),
            'description': description,
            'source': source,
            'registered_date': datetime.now().isoformat(),
            'tags': tags or [],
            'metadata': metadata or {}
        }
        
        # Add file information if file exists
        if path.exists():
            dataset_info['file_info'] = {
                'size_bytes': path.stat().st_size,
                'size_mb': path.stat().st_size / 1024 / 1024,
                'modified_date': datetime.fromtimestamp(path.stat().st_mtime).isoformat()
            }
        
        # Update or add dataset
        existing_idx = next((i for i, d in enumerate(self.catalog['datasets']) 
                           if d['name'] == name), None)
        
        if existing_idx is not None:
            self.catalog['datasets'][existing_idx] = dataset_info
            print(f"✅ Updated dataset '{name}' in catalog")
        else:
            self.catalog['datasets'].append(dataset_info)
            print(f"✅ Registered new dataset '{name}' in catalog")
        
        self._save_catalog()
    
    def list_datasets(self, tag=None):
        """List all datasets or filter by tag."""
        datasets = self.catalog['datasets']
        
        if tag:
            datasets = [d for d in datasets if tag in d.get('tags', [])]
        
        if not datasets:
            print(f"No datasets found{' with tag: ' + tag if tag else ''}")
            return
        
        print(f"\n📚 Data Catalog{' (tag: ' + tag + ')' if tag else ''}:")
        print("=" * 100)
        print(f"{'Name':<25} {'Source':<20} {'Size (MB)':<12} {'Tags':<20}")
        print("-" * 100)
        
        for d in datasets:
            size_mb = d.get('file_info', {}).get('size_mb', 0)
            tags_str = ', '.join(d.get('tags', []))
            print(f"{d['name']:<25} {d['source']:<20} {size_mb:>10.2f}  {tags_str:<20}")
        
        print("-" * 100)
        print(f"Total datasets: {len(datasets)}")
    
    def get_dataset(self, name):
        """Get information about a specific dataset."""
        dataset = next((d for d in self.catalog['datasets'] if d['name'] == name), None)
        
        if dataset is None:
            print(f"❌ Dataset '{name}' not found in catalog")
            return None
        
        print(f"\n📊 Dataset: {dataset['name']}")
        print("=" * 60)
        print(f"Description: {dataset['description']}")
        print(f"Source: {dataset['source']}")
        print(f"Path: {dataset['path']}")
        print(f"Registered: {dataset['registered_date'][:19]}")
        print(f"Tags: {', '.join(dataset.get('tags', []))}")
        
        if 'file_info' in dataset:
            print(f"\nFile Info:")
            print(f"  Size: {dataset['file_info']['size_mb']:.2f} MB")
            print(f"  Modified: {dataset['file_info']['modified_date'][:19]}")
        
        return dataset


# Example usage
catalog_file = TEST_DIR / 'data_catalog.json'
catalog = DataCatalog(catalog_file)

# Register some datasets
catalog.register_dataset(
    name='bike_availability_v1',
    path=TEST_DIR / 'bike_availability_v1_2026-01-15_143022.parquet',
    description='Initial bike availability dataset with 30 days of data',
    source='CityBikes API',
    tags=['raw', 'bike', 'amsterdam']
)

catalog.register_dataset(
    name='bike_availability_v2',
    path=TEST_DIR / 'bike_availability_v2_2026-01-15_143023.parquet',
    description='Bike availability with total_capacity column added',
    source='CityBikes API (processed)',
    tags=['processed', 'bike', 'amsterdam', 'featured']
)

# List all datasets
catalog.list_datasets()

# List datasets by tag
print("\n")
catalog.list_datasets(tag='processed')

# Get specific dataset info
catalog.get_dataset('bike_availability_v2')

---

## 💾 Part 9: Save Best Practices Example

Here's a complete example of saving data with all best practices.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. COMPLETE SAVE EXAMPLE
# ═══════════════════════════════════════════════════════════

def save_dataset_with_best_practices(df, dataset_name, source, description, 
                                    output_dir, version_manager=None, 
                                    data_catalog=None):
    """
    Save dataset following all best practices.
    
    This function:
    1. Saves data in Parquet format
    2. Creates documentation
    3. Manages versioning
    4. Updates data catalog
    5. Returns file paths
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save data in Parquet format
    timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
    data_file = output_dir / f"{dataset_name}_{timestamp}.parquet"
    df.to_parquet(data_file, index=False)
    print(f"✅ Saved data: {data_file.name}")
    
    # 2. Create and save documentation
    doc = create_data_documentation(df, dataset_name, source, description)
    doc_file = data_file.with_suffix('.metadata.json')
    with open(doc_file, 'w') as f:
        json.dump(doc, f, indent=2, default=str)
    print(f"✅ Saved documentation: {doc_file.name}")
    
    # 3. Version management (if provided)
    if version_manager:
        version_manager.save_version(df, dataset_name, description)
        print(f"✅ Version tracked")
    
    # 4. Update data catalog (if provided)
    if data_catalog:
        data_catalog.register_dataset(
            name=dataset_name,
            path=data_file,
            description=description,
            source=source,
            tags=['processed', 'documented']
        )
        print(f"✅ Catalog updated")
    
    # 5. Create README if it doesn't exist
    readme_file = output_dir / 'README.md'
    if not readme_file.exists():
        readme_content = f"""# Processed Data

This directory contains processed datasets ready for analysis.

## Latest Datasets

### {dataset_name}
- **Source**: {source}
- **Description**: {description}
- **Created**: {timestamp}
- **Rows**: {len(df):,}
- **Columns**: {len(df.columns)}

## File Naming Convention

Files follow the pattern: `{{dataset_name}}_{{YYYY-MM-DD_HHMMSS}}.parquet`

## Documentation

Each dataset has an accompanying `.metadata.json` file with:
- Column descriptions and statistics
- Data quality information
- Source and transformation details
- Date range and sample size

Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
        with open(readme_file, 'w') as f:
            f.write(readme_content)
        print(f"✅ Created README: {readme_file.name}")
    
    return {
        'data_file': data_file,
        'doc_file': doc_file,
        'readme_file': readme_file
    }


# Example: Save with all best practices
print("💾 Saving dataset with best practices...")
print("=" * 60)

files = save_dataset_with_best_practices(
    df=df_sample,
    dataset_name='bike_availability_clean',
    source='CityBikes API + Open-Meteo API',
    description='Cleaned and merged bike availability with weather data',
    output_dir=TEST_DIR,
    version_manager=vm,
    data_catalog=catalog
)

print("\n" + "=" * 60)
print("📁 Files created:")
for key, path in files.items():
    print(f"  • {key}: {path.name}")

---

### 🧠 Task 9.1: Build Complete Workflow (30% Scaffolding)

**Your Task**: Create an end-to-end data storage workflow for a new scenario.

**What you'll learn**:
- Integrating all storage concepts
- Making architectural decisions
- Building production-ready solutions

**Scenario**: You've collected 7 days of bike data for 3 stations. Build a complete storage solution.

**Requirements**:
1. Generate sample data (3 stations × 7 days × 24 hours)
2. Choose and justify file format
3. Implement versioning strategy
4. Create complete documentation
5. Set up logical directory structure
6. Write decision summary

**Minimal scaffolding** - Design your own approach:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 9.1: COMPLETE WORKFLOW (30% SCAFFOLDING)
# ═══════════════════════════════════════════════════════════

"""
Build an end-to-end storage workflow for new bike data collection.

Your workflow should:
1. Generate realistic sample data
2. Choose optimal file format with justification
3. Implement versioning
4. Create documentation
5. Set up directory structure
6. Generate summary report
"""

# TODO: Step 1 - Generate sample data
# Create data for 3 stations × 7 days × 24 hours = 504 records

# TODO: Step 2 - Choose file format
# Consider: data size, update frequency, read patterns
# Justify your choice

# TODO: Step 3 - Implement versioning
# Use your extended DataVersionManager

# TODO: Step 4 - Create documentation
# README.md + metadata.json

# TODO: Step 5 - Directory structure
# Create organized folder hierarchy

# TODO: Step 6 - Summary report
# Document all decisions and trade-offs

# --- Your implementation here ---

# Final summary template:
print("""
╔════════════════════════════════════════════════════════════╗
║          DATA STORAGE WORKFLOW SUMMARY                     ║
╚════════════════════════════════════════════════════════════╝

📊 Dataset: 3 stations × 7 days × 24 hours = ??? records

🗂️  Format Choice: ???
   Reasoning: ???

📁 Directory Structure:
   data/
     ├── processed/
     │   ├── bike_weekly_v1.???
     │   └── metadata/
     └── docs/

✅ Implementation Complete:
   - Data saved with version v1
   - Documentation created
   - Metadata tracked
   - Directory structure established
""")

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 9.1: COMPLETE WORKFLOW
# ═══════════════════════════════════════════════════════════

# Step 1: Generate sample data
# TODO: Create 3 stations, 7 days of hourly data
# Consider: What columns should it have?

# Step 2: Choose format and save
# TODO: Decide format (CSV/JSON/Parquet) and justify
# Consider: Data size, read frequency, collaborators

# Step 3: Implement versioning
# TODO: Use DataVersionManager or create custom approach
# Consider: How often will this data update?

# Step 4: Create documentation  
# TODO: README, metadata JSON, column descriptions
# Consider: What does someone need to know to use this data?

# Step 5: Directory structure
# TODO: Organize files logically
# Suggested:
#   my_workflow/
#     data/
#     metadata/
#     docs/
#     README.md

# Step 6: Summary report
print("=" * 70)
print("📊 WORKFLOW SUMMARY REPORT")
print("=" * 70)
print("\n1. FORMAT CHOICE:")
print("   Selected: _______")
print("   Reason: _______")
print("\n2. VERSIONING STRATEGY:")
print("   Approach: _______")
print("   Reason: _______")
print("\n3. DOCUMENTATION:")
print("   Files created: _______")
print("   Key information included: _______")
print("\n4. TRADE-OFFS:")
print("   Pros of my approach: _______")
print("   Cons of my approach: _______")
print("\n5. PRODUCTION READINESS:")
print("   Ready for production? _______")
print("   What would you improve? _______")

---

## 🔥 Part 9a: Optional Advanced Challenges

**For advanced learners**: These challenges explore production-grade storage patterns!

### Challenge 6.1: DVC Integration 🔄

**Task**: Set up Data Version Control (DVC) for the project.

**Requirements**:
- Install DVC: `pip install dvc`
- Initialize DVC in the project
- Track the processed data directory with DVC
- Create a DVC pipeline for data processing
- Push data to remote storage (local for testing)

**Hints**:
```bash
dvc init
dvc add data/processed
git add data/processed.dvc .gitignore
dvc remote add -d local_remote /tmp/dvc-storage
dvc push
```

**Learning**: Enterprise-level data versioning

---

### Challenge 6.2: Cloud Storage Integration ☁️

**Task**: Implement cloud storage (AWS S3 or Google Cloud Storage) for large datasets.

**Requirements**:
- Set up credentials for cloud provider
- Create function to upload/download from cloud
- Implement caching: check local first, download if needed
- Add cloud paths to data catalog
- Handle connection errors gracefully

**Hints**:
- Use `boto3` for AWS or `google-cloud-storage`
- Consider `s3fs` or `gcsfs` for Parquet direct reading
- Cache metadata locally, data in cloud

**Learning**: Scalable storage for production systems

---

### Challenge 6.3: Streaming Data Storage 📡

**Task**: Design a storage system for real-time streaming bike data.

**Requirements**:
- Create append-only storage structure
- Implement time-partitioned files (hourly/daily)
- Build efficient query interface for date ranges
- Handle late-arriving data
- Implement data compaction strategy

**Suggested structure**:
```
data/
  streaming/
    2026/
      01/
        bike_data_2026-01-01_00.parquet
        bike_data_2026-01-01_01.parquet
        ...
```

**Hints**:
- Use Parquet partitioning: `df.to_parquet('data/streaming', partition_cols=['year', 'month', 'day'])`
- Consider: How to query last 24 hours efficiently?
- Think about: When to compact small files?

**Learning**: Real-time data engineering patterns

---

### Challenge 6.4: Data Lineage Tracker 📊

**Task**: Build a system to track data transformations and dependencies.

**Requirements**:
- Create `DataLineage` class that records:
  - Input datasets used
  - Transformations applied
  - Output datasets created
  - Timestamps and user
- Generate visual lineage graph
- Export lineage as JSON
- Integrate with your save workflow

**Example tracking**:
```python
lineage = DataLineage()
lineage.add_input('bike_api_data.csv')
lineage.add_transformation('remove_nulls', params={'threshold': 0.1})
lineage.add_transformation('add_bikeability_score')
lineage.add_output('bike_processed_v2.parquet')
lineage.visualize()  # Creates flowchart
```

**Hints**:
- Use `graphviz` or `networkx` for visualization
- Store lineage metadata with datasets
- Consider: How to trace data quality issues back to source?

**Learning**: Data governance and reproducibility

---

## 📝 Part 10: Summary

### What You've Learned ✅

In this notebook, you:
1. ✅ Compared file formats (CSV, JSON, Parquet, Feather)
2. ✅ Measured storage size and I/O performance
3. ✅ Implemented data versioning with metadata
4. ✅ Created comprehensive data documentation
5. ✅ Set up .gitignore best practices
6. ✅ Built a data catalog system
7. ✅ Learned complete save workflow with all best practices

### Key Takeaways 💡

1. **Format matters** - Parquet is best for processed data
2. **Version everything** - Track changes over time
3. **Document thoroughly** - Future you will thank current you
4. **Never commit large files** - Use .gitignore properly
5. **Catalog your data** - Make datasets discoverable
6. **Automate workflows** - Create reusable save functions

### Best Practices Checklist ✅

- [ ] Store raw data in original format (immutable)
- [ ] Use Parquet for processed data (compressed, fast)
- [ ] Version datasets with clear naming
- [ ] Create metadata for every dataset
- [ ] Update .gitignore to exclude large files
- [ ] Maintain a data catalog
- [ ] Document transformations in README files
- [ ] Back up important data

### Next Steps 🚀

Now that you understand storage patterns, proceed to:
- **M2_04_merge_datasets.ipynb** - Combine bike and weather data with proper storage

### 🧠 Reflection Questions

1. **When would you choose CSV** over Parquet despite its larger size?
2. **How would you handle** versioning for a dataset that updates daily?
3. **What information** should always be in data documentation?
4. **How would you organize** data for a team of 10 data scientists?

**Write your reflections below** ⬇️

### My Reflections

[Your thoughts here]

---

## 📚 References

- [Pandas I/O Tools](https://pandas.pydata.org/docs/user_guide/io.html)
- [Parquet Format Documentation](https://parquet.apache.org/docs/)
- [Data Version Control (DVC)](https://dvc.org/)
- [Git LFS Documentation](https://git-lfs.github.com/)
- [Data Management Best Practices](https://the-turing-way.netlify.app/reproducible-research/rdm.html)

---

**🎉 Congratulations!** You've successfully completed M2_03 - Data Storage Patterns!